# NBA Player Performance Prediction

## Problem Statement
Can we explain or estimate an NBA player's scoring output (or role) for a season based on usage, efficiency, and player characteristics?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Dataset

This data set contains over 20 years of data on each player who has been a part of an NBA teams' roster. It contains demographics such as age, height, weight, and place of birth as well as game statistics like average points, assists, rebounds, and games played. It also has draft year, round, team played for, and other biographicals.

Dataset credits to Kaggle user Justinas Cirtautas.

In [2]:
nba_data = pd.read_csv("../data/raw/nba_player_stats.csv")

nba_data.head()

,Unnamed: 0,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,...,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season
0,0,Randy Livingston,HOU,22.0,193.04,94.800728,Louisiana State,USA,1996,2,...,3.9,1.5,2.4,0.3,0.042,0.071,0.169,0.487,0.248,1996-97
1,1,Gaylon Nickerson,WAS,28.0,190.50,86.182480,Northwestern Oklahoma,USA,1994,2,...,3.8,1.3,0.3,8.9,0.030,0.111,0.174,0.497,0.043,1996-97
2,2,George Lynch,VAN,26.0,203.20,103.418976,North Carolina,USA,1993,1,...,8.3,6.4,1.9,-8.2,0.106,0.185,0.175,0.512,0.125,1996-97
3,3,George McCloud,LAL,30.0,203.20,102.058200,Florida State,USA,1989,1,...,10.2,2.8,1.7,-2.7,0.027,0.111,0.206,0.527,0.125,1996-97
4,4,George Zidek,DEN,23.0,213.36,119.748288,UCLA,USA,1995,1,...,2.8,1.7,0.3,-14.1,0.102,0.169,0.195,0.500,0.064,1996-97


In [3]:
# Prediction target feature
y = nba_data.pts

# Only use significant features. Identifiers aren't very important right now
feature_names = ['age', 'player_height', 'player_weight', 'gp', 'reb', 'ast', 'oreb_pct', 'dreb_pct', 'ast_pct']
X = nba_data[feature_names]

## Exploratory Data Analysis

In [11]:
# machine learning imports

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [5]:
# split data 80:20 for training and testing
train_X, val_X, train_y, val_y = train_test_split(X, y, test_size=0.2, random_state=5)

In [6]:
# functions to get mean absolute error based on train test split data and argument:value
def create_model(argument, value):
    model = RandomForestRegressor(
        **{argument: value},
        random_state=5)
    return model

def score_model(model, X_t=train_X, X_v=val_X, y_t=train_y, y_v=val_y):
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    return mean_absolute_error(y_v, preds)

In [7]:
#n_estimators=100
# for estimators in [50, 100, 250, 500]:
#     model= create_model("n_estimators", estimators)
#     print(estimators, score_model(model))
#
# #max_depth=None
# for depth in [5, 10, 15, 20]:
#     model= create_model("max_depth", depth)
#     print(depth, score_model(model))
#
# #min_samples_split=2
# for sample_splits in [2, 5, 10, 20]:
#     model= create_model("min_samples_split", sample_splits)
#     print(sample_splits, score_model(model))
#
# #max_leaf_nodes=None
# for nodes in [50, 100, 250, 500]:
#     model= create_model("max_leaf_nodes", nodes)
#     print(nodes, score_model(model))
#
# #min_samples_leaf=1
# for sample_leaf in [1, 2, 5, 10]:
#     model = create_model("min_samples_leaf", sample_leaf)
#     print(sample_leaf, score_model(model))

Best Individual Argument: min_samples_leaf=2, all other defaults

Next: Missing Values

In [12]:
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=10, random_state=5)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

# Drop columns with missing values
cols_with_missing = [col for col in train_X.columns if train_X[col].isnull().any()]

red_train_X = train_X.drop(cols_with_missing, axis=1)
red_val_X = val_X.drop(cols_with_missing, axis=1)

print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(red_train_X, red_val_X, train_y, val_y))


# Impute missing values with means
my_imputer = SimpleImputer()
imp_train_X = pd.DataFrame(my_imputer.fit_transform(train_X))
imp_val_X = pd.DataFrame(my_imputer.transform(val_X))

imp_train_X.columns = train_X.columns
imp_val_X.columns = val_X.columns

print("MAE from Approach 2 (Imputation):")
print(score_dataset(imp_train_X, imp_val_X, train_y, val_y))

MAE from Approach 1 (Drop columns with missing values):
1.7103308680420397
MAE from Approach 2 (Imputation):
1.7103308680420397
